In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd

res_dir = '../Results/'
base_names = ['Random Forest', 'Logistic Regression', 'Decision Tree', 'XGBoost', 'CatBoost', 'AdaBoost', 'LightGBM', 'Support Vector Machine', 'K-Nearest Neighbors', 'Extra Trees', 'Gradient Boosting', 'Naive Bayes']
rank_by = 'AUC'

results_df = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob(res_dir + 'CV_Fold_Results_*.pkl'))], ignore_index=True)
results_df = results_df[results_df['Classifier'].isin(base_names)]
results_df.shape

(10140, 10)

In [2]:
auc_tables = {}
error_tables = {}
for ratio, g in results_df.groupby('Ratio'):
    auc_tables[ratio] = g.groupby(['No', 'Classifier'])['AUC'].mean().unstack()
    error_tables[ratio] = 1 - g.groupby(['No', 'Classifier'])['Accuracy'].mean().unstack()
for ratio in auc_tables:
    print('Ratio', ratio)
    print(auc_tables[ratio].round(4).to_string())

Ratio 0.7
Classifier  AdaBoost  CatBoost  Decision Tree  Extra Trees  Gradient Boosting  K-Nearest Neighbors  LightGBM  Logistic Regression  Naive Bayes  Random Forest  Support Vector Machine  XGBoost
No                                                                                                                                                                                            
1             0.8345    0.8281         0.7369       0.8094             0.8238               0.7422    0.8150               0.7893       0.7593         0.8348                  0.7579   0.8387
2             0.8145    0.8101         0.8014       0.8069             0.8197               0.7662    0.8212               0.7801       0.7540         0.8232                  0.7725   0.8292
3             0.7136    0.6622         0.6217       0.6665             0.6566               0.6643    0.6893               0.7435       0.7418         0.6953                  0.7309   0.6867
4             0.8009    0.8153     

In [3]:
for ratio in error_tables:
    print('Ratio', ratio)
    print(error_tables[ratio].round(4).to_string())

Ratio 0.7
Classifier  AdaBoost  CatBoost  Decision Tree  Extra Trees  Gradient Boosting  K-Nearest Neighbors  LightGBM  Logistic Regression  Naive Bayes  Random Forest  Support Vector Machine  XGBoost
No                                                                                                                                                                                            
1             0.2492    0.2687         0.3021       0.2793             0.2590               0.2948    0.2679               0.2875       0.3037         0.2635                  0.3083   0.2582
2             0.2487    0.2726         0.2883       0.2905             0.2866               0.3063    0.2499               0.3058       0.3185         0.2663                  0.3023   0.2499
3             0.3088    0.3522         0.3589       0.3415             0.3560               0.3252    0.3172               0.3615       0.2939         0.3425                  0.3730   0.3322
4             0.2646    0.2651     

In [4]:
top_info = {}
for ratio, g in results_df.groupby('Ratio'):
    table = g.groupby(['No', 'Classifier'])[rank_by].mean().unstack()
    entry = {}
    scores = table.mean().sort_values(ascending=False)
    entry['global'] = {str(t): scores.index[:t].tolist() for t in [3, 5, 7]}
    entry['global']['scores'] = {n: float(v) for n, v in scores.items()}
    for number, row in table.iterrows():
        row = row.sort_values(ascending=False)
        entry['Dataset_' + str(number)] = {str(t): row.index[:t].tolist() for t in [3, 5, 7]}
        entry['Dataset_' + str(number)]['scores'] = {n: float(v) for n, v in row.items()}
    top_info[str(ratio)] = entry
    print('Ratio', ratio, 'Best seven classifiers:', entry['global']['7'])
json.dump(top_info, open(res_dir + 'top_classifiers.json', 'w'), indent=2)

Ratio 0.7 Best seven classifiers: ['XGBoost', 'Random Forest', 'AdaBoost', 'LightGBM', 'CatBoost', 'Logistic Regression', 'Gradient Boosting']
Ratio 0.8 Best seven classifiers: ['AdaBoost', 'LightGBM', 'XGBoost', 'Random Forest', 'CatBoost', 'Gradient Boosting', 'Extra Trees']


In [5]:
mean_auc = pd.DataFrame({ratio: auc_tables[ratio].mean() for ratio in auc_tables}).sort_values(by=list(auc_tables.keys())[0], ascending=False)
mean_auc

,0.7,0.8
Classifier,,
XGBoost,0.792876,0.823661
Random Forest,0.790993,0.822348
AdaBoost,0.790868,0.829910
LightGBM,0.779278,0.825531
CatBoost,0.778912,0.820773
Logistic Regression,0.778572,0.778448
Gradient Boosting,0.773499,0.816150
Extra Trees,0.766030,0.791190
Support Vector Machine,0.757564,0.770999
